# Chapter 7 — The Laplace Transform and Its Applications

This interactive notebook accompanies the chapter on one-sided, bilateral, and symmetric Laplace transforms. It combines symbolic computation, numerical experiments, plots, short proofs, direct verification, and self-assessment.

The one-sided Laplace transform is

$$
\mathcal{L}\{f\}(s)=F(s)=\int_0^\infty e^{-st}f(t)\,dt.
$$

The algebraic expression for $F$ is only part of the answer: its **region of convergence** must also be stated. When a transform method is used formally to discover a solution, the final candidate must be checked directly in the original equation.

> Run the cells from top to bottom. Change the controls, press the buttons, and compare the computational evidence with the mathematical statements.

## 0. Computational environment

The notebook uses NumPy and SciPy for numerical work, SymPy for symbolic transforms, Matplotlib for visualization, and `ipywidgets` for interactivity.

For a numerical approximation on $[0,T]$ we use

$$
\mathcal{L}_T\{f\}(s)=\int_0^T e^{-st}f(t)\,dt,
$$

and compare it with either the exact transform or a larger truncation. The helper functions below also format symbolic results through MathJax.

In [ ]:
# Core imports used throughout the notebook.
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import importlib.util
import subprocess
import sys

# Install only packages that are missing. Standard Jupyter/Colab environments
# usually skip this block, while a fresh local kernel remains easy to set up.
requirements = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "sympy": "sympy",
    "ipywidgets": "ipywidgets",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing educational dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

from scipy.integrate import quad, cumulative_trapezoid
from scipy.special import gamma as gamma_fn, beta as beta_fn, erf
from scipy.stats import binom
from IPython.display import display, Math, Markdown, clear_output
import ipywidgets as widgets

# Global plotting style and colors.
plt.rcParams.update({
    "figure.figsize": (9, 4.8),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

BLUE = "#2563eb"
ORANGE = "#f97316"
GREEN = "#16a34a"
RED = "#dc2626"
PURPLE = "#7c3aed"
STYLE = {"description_width": "initial"}
WIDE = widgets.Layout(width="520px")

t_sym, s_sym = sp.symbols("t s", positive=True)

def show_math(latex_source):
    """Render a LaTeX string with MathJax."""
    display(Math(latex_source))

def close_plot():
    """Apply a compact layout and display the current Matplotlib figure."""
    plt.tight_layout()
    plt.show()

display(widgets.HTML(
    "<div style='padding:10px 14px;border-left:5px solid #2563eb;"
    "background:#eff6ff'><b>Ready.</b> Use the controls in each laboratory below.</div>"
))

## 1. Absolute convergence and the right half-plane

If $f$ is of exponential order $a$, then $|f(t)|\leq Me^{at}$ for large $t$. Hence

$$
|e^{-st}f(t)|\leq M e^{-(\operatorname{Re}s-a)t},
$$

so the Laplace integral converges absolutely whenever $\operatorname{Re}s>a$.

For $f(t)=e^{at}\cos(bt)$ the exact transform is

$$
F(s)=\frac{s-a}{(s-a)^2+b^2},
\qquad \operatorname{Re}s>a.
$$

Move $\sigma=\operatorname{Re}s$ across the boundary $\sigma=a$. The first plot shows the damped integrand, while the second shows how the truncated integral changes with $T$.

In [ ]:
def convergence_lab(a=1.0, b=2.0, sigma=1.6):
    """Explore the convergence boundary for exp(a t) cos(b t)."""
    time = np.linspace(0.0, 18.0, 1800)
    integrand = np.exp(np.clip((a - sigma) * time, -700, 40)) * np.cos(b * time)
    truncations = np.linspace(0.25, 18.0, 160)
    partial = np.array([
        quad(lambda u: np.exp((a - sigma) * u) * np.cos(b * u), 0, T, limit=300)[0]
        for T in truncations
    ])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(time, integrand, color=BLUE)
    axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set(xlabel="$t$", ylabel="$e^{-\\sigma t}f(t)$", title="Damped integrand")

    axes[1].plot(truncations, partial, color=ORANGE, label="$\\mathcal{L}_T\\{f\\}(\\sigma)$")
    if sigma > a:
        exact = (sigma - a) / ((sigma - a)**2 + b**2)
        axes[1].axhline(exact, color=GREEN, ls="--", label=f"exact = {exact:.5f}")
        status = "Absolute convergence:  σ > a"
        color = GREEN
    else:
        status = "Outside the absolute-convergence half-plane"
        color = RED
    axes[1].set(xlabel="$T$", ylabel="truncated integral", title=status)
    axes[1].legend()
    close_plot()

controls = {
    "a": widgets.FloatSlider(value=1.0, min=-1.0, max=2.0, step=0.1, description="growth rate a", style=STYLE),
    "b": widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="frequency b", style=STYLE),
    "sigma": widgets.FloatSlider(value=1.6, min=-0.5, max=4.0, step=0.1, description="σ = Re(s)", style=STYLE),
}
display(widgets.VBox(list(controls.values())))
display(widgets.interactive_output(convergence_lab, controls))

## 2. Basic transform pairs

The most frequently used pairs include

$$
\mathcal{L}\{1\}=\frac1s,
\qquad
\mathcal{L}\{t^n\}=\frac{n!}{s^{n+1}},
$$

$$
\mathcal{L}\{e^{at}\}=\frac1{s-a},
\qquad
\mathcal{L}\{\sin(bt)\}=\frac{b}{s^2+b^2},
\qquad
\mathcal{L}\{\cos(bt)\}=\frac{s}{s^2+b^2}.
$$

The laboratory evaluates both sides at a real $s$ in the appropriate half-plane. Numerical quadrature is evidence, while the identity is established analytically by integration.

In [ ]:
def transform_pair_lab(pair="t^n", parameter=2.0, s_value=3.0):
    """Compare a standard transform pair with numerical quadrature."""
    n = max(0, int(round(parameter)))
    if pair == "1":
        f = lambda t: 1.0
        exact = 1.0 / s_value
        formula = r"\mathcal{L}\{1\}(s)=\frac{1}{s}"
        boundary = 0.0
    elif pair == "exp(a t)":
        a = parameter
        f = lambda t: np.exp(a * t)
        exact = 1.0 / (s_value - a)
        formula = rf"\mathcal{{L}}\{{e^{{{a:.2g}t}}\}}(s)=\frac{{1}}{{s-{a:.2g}}}"
        boundary = a
    elif pair == "t^n":
        f = lambda t: t**n
        exact = float(sp.factorial(n)) / s_value**(n + 1)
        formula = rf"\mathcal{{L}}\{{t^{n}\}}(s)=\frac{{{n}!}}{{s^{{{n+1}}}}}"
        boundary = 0.0
    elif pair == "sin(b t)":
        b = max(0.1, abs(parameter))
        f = lambda t: np.sin(b * t)
        exact = b / (s_value**2 + b**2)
        formula = rf"\mathcal{{L}}\{{\sin({b:.2g}t)\}}(s)=\frac{{{b:.2g}}}{{s^2+{b:.2g}^2}}"
        boundary = 0.0
    else:
        b = max(0.1, abs(parameter))
        f = lambda t: np.cos(b * t)
        exact = s_value / (s_value**2 + b**2)
        formula = rf"\mathcal{{L}}\{{\cos({b:.2g}t)\}}(s)=\frac{{s}}{{s^2+{b:.2g}^2}}"
        boundary = 0.0

    clear_output(wait=True)
    show_math(formula)
    if s_value <= boundary:
        display(widgets.HTML("<b style='color:#dc2626'>Choose s to the right of the convergence boundary.</b>"))
        return
    # Combine exponential factors before evaluation to avoid overflow in exp(a t).
    if pair == "exp(a t)":
        numeric, error = quad(lambda u: np.exp(-(s_value-parameter)*u), 0, np.inf, limit=300)
    else:
        numeric, error = quad(lambda u: np.exp(-s_value * u) * f(u), 0, np.inf, limit=300)
    show_math(rf"s={s_value:.3g}:\quad F_{{\rm num}}={numeric:.10g},\quad F_{{\rm exact}}={exact:.10g}")
    print(f"absolute error = {abs(numeric-exact):.3e}   |   quadrature estimate = {error:.3e}")

pair_box = widgets.Dropdown(options=["1", "exp(a t)", "t^n", "sin(b t)", "cos(b t)"], value="t^n", description="function", style=STYLE)
param_box = widgets.FloatSlider(value=2.0, min=-1.0, max=6.0, step=1.0, description="a, b, or n", style=STYLE)
s_box = widgets.FloatSlider(value=3.0, min=0.2, max=8.0, step=0.1, description="real s", style=STYLE)
output = widgets.Output()
button = widgets.Button(description="Compare transform", button_style="primary", icon="check")

def run_pair(_):
    with output:
        transform_pair_lab(pair_box.value, param_box.value, s_box.value)

button.on_click(run_pair)
display(widgets.VBox([pair_box, param_box, s_box, button, output]))
run_pair(None)

## 3. User-defined function: symbolic Laplace transform

Enter a function of $t$ using SymPy notation. The cell asks SymPy for the transform together with its convergence condition:

$$
(F(s),a,C)=\operatorname{laplace\_transform}(f(t),t,s),
$$

where $\operatorname{Re}s>a$ is the returned convergence half-plane and $C$ contains any additional parameter conditions.

Examples of accepted syntax are `t**2*exp(-3*t)*sin(2*t)`, `Heaviside(t-2)*(t-2)`, `sin(t)/t`, and `exp(-t**2)`. The result is typeset by MathJax. Always record the convergence information, not only the algebraic expression.

In [ ]:
# Symbols and safe mathematical names made available to the input parser.
t = sp.symbols("t", positive=True, real=True)
s = sp.symbols("s")
parser_names = {
    "t": t, "s": s, "exp": sp.exp, "sin": sp.sin, "cos": sp.cos,
    "sinh": sp.sinh, "cosh": sp.cosh, "sqrt": sp.sqrt, "log": sp.log,
    "Heaviside": sp.Heaviside, "gamma": sp.gamma, "erf": sp.erf,
    "pi": sp.pi,
}

preset_functions = {
    "Polynomial × exponential × sine": "t**2*exp(-3*t)*sin(2*t)",
    "Delayed ramp": "Heaviside(t-2)*(t-2)",
    "Sinc": "sin(t)/t",
    "Gaussian": "exp(-t**2)",
    "Custom": "exp(-t)*cos(3*t)",
}

preset = widgets.Dropdown(options=list(preset_functions), value="Polynomial × exponential × sine", description="example", style=STYLE)
function_input = widgets.Text(value=preset_functions[preset.value], description="f(t) =", style=STYLE, layout=WIDE)
compute_button = widgets.Button(description="Compute Laplace transform", button_style="success", icon="calculator")
custom_output = widgets.Output()

def load_preset(change):
    if change["new"] != "Custom":
        function_input.value = preset_functions[change["new"]]

def compute_custom_laplace(_):
    with custom_output:
        clear_output(wait=True)
        try:
            expression = sp.sympify(function_input.value, locals=parser_names)
            transform, convergence_abscissa, condition = sp.laplace_transform(expression, t, s, noconds=False)
            display(widgets.HTML("<h4 style='color:#1d4ed8'>MathJax result</h4>"))
            show_math(r"f(t)=" + sp.latex(expression))
            show_math(r"\mathcal{L}\{f\}(s)=" + sp.latex(sp.simplify(transform)))
            if convergence_abscissa is not None:
                show_math(r"\operatorname{Re}(s)>" + sp.latex(convergence_abscissa))
            show_math(r"\text{Additional condition: }" + sp.latex(condition))
            display(widgets.HTML(
                "<div style='padding:8px;background:#f0fdf4;border-left:4px solid #16a34a'>"
                "The transform expression and its convergence information should be reported together.</div>"
            ))
        except Exception as error:
            display(widgets.HTML(f"<b style='color:#dc2626'>Input error:</b> {error}"))
            print("Try syntax such as: t**2*exp(-3*t)*sin(2*t)")

preset.observe(load_preset, names="value")
compute_button.on_click(compute_custom_laplace)
display(widgets.VBox([preset, function_input, compute_button, custom_output]))
compute_custom_laplace(None)

## 4. Power series: when termwise transformation fails

Under a suitable integrable majorant, one may write

$$
f(t)=\sum_{n=0}^{\infty}a_nt^n
\quad\Longrightarrow\quad
\mathcal{L}\{f\}(s)=\sum_{n=0}^{\infty}\frac{a_n n!}{s^{n+1}}.
$$

Pointwise convergence alone is insufficient. Although

$$
e^{-t^2}=\sum_{n=0}^{\infty}\frac{(-1)^nt^{2n}}{n!},
$$

the formally transformed series has terms

$$
b_n(s)=\frac{(-1)^n(2n)!}{n!s^{2n+1}},
$$

whose magnitudes eventually grow for every fixed $s\ne0$. The ratio is $|b_{n+1}/b_n|=2(2n+1)/|s|^2$.

In [ ]:
def gaussian_series_warning(s_value=4.0, terms=18):
    """Visualize divergence of the formally transformed Gaussian series."""
    n = np.arange(terms)
    log_terms = np.array([
        float(sp.loggamma(2*int(k) + 1) - sp.loggamma(int(k) + 1) - (2*int(k) + 1)*sp.log(s_value))
        for k in n
    ])
    ratios = 2 * (2*n[:-1] + 1) / s_value**2

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(n, log_terms / np.log(10), "o-", color=RED)
    axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set(xlabel="$n$", ylabel="$\\log_{10}|b_n(s)|$", title="Terms do not tend to zero")
    axes[1].plot(n[:-1], ratios, "o-", color=PURPLE)
    axes[1].axhline(1, color="black", ls="--", label="ratio = 1")
    axes[1].set(xlabel="$n$", ylabel="$|b_{n+1}/b_n|$", title="Ratio test diagnosis")
    axes[1].legend()
    close_plot()

display(widgets.interactive(
    gaussian_series_warning,
    s_value=widgets.FloatSlider(value=4.0, min=0.8, max=10.0, step=0.2, description="fixed s", style=STYLE),
    terms=widgets.IntSlider(value=18, min=6, max=35, step=1, description="terms", style=STYLE),
))

## 5. Shifts and scaling

If $F(s)=\mathcal{L}\{f\}(s)$, then

$$
\mathcal{L}\{e^{at}f(t)\}(s)=F(s-a),
$$

$$
\mathcal{L}\{f(t-c)H(t-c)\}(s)=e^{-cs}F(s),
$$

and, for $q>0$,

$$
\mathcal{L}\{f(qt)\}(s)=\frac1qF\!\left(\frac{s}{q}\right).
$$

The control below delays $f(t)=\sin(bt)$ by $c$. The time-domain signal starts at $t=c$, while its transform acquires the factor $e^{-cs}$.

In [ ]:
def delay_lab(b=2.0, c=1.5, s_value=2.5):
    """Verify the second-shift theorem numerically."""
    time = np.linspace(0, 10, 1000)
    delayed = np.where(time >= c, np.sin(b * (time - c)), 0.0)
    exact = np.exp(-c * s_value) * b / (s_value**2 + b**2)
    numeric = quad(
        lambda u: np.exp(-s_value*u) * np.sin(b*(u-c)), c, np.inf, limit=300
    )[0]

    plt.plot(time, delayed, color=BLUE, lw=2)
    plt.axvline(c, color=ORANGE, ls="--", label=f"delay c={c:.2f}")
    plt.xlabel("$t$")
    plt.ylabel("delayed signal")
    plt.title(f"numeric transform = {numeric:.7f}   |   exact = {exact:.7f}")
    plt.legend()
    close_plot()

display(widgets.interactive(
    delay_lab,
    b=widgets.FloatSlider(value=2.0, min=0.3, max=5.0, step=0.1, description="frequency b", style=STYLE),
    c=widgets.FloatSlider(value=1.5, min=0.0, max=5.0, step=0.1, description="delay c", style=STYLE),
    s_value=widgets.FloatSlider(value=2.5, min=0.2, max=6.0, step=0.1, description="real s", style=STYLE),
))

## 6. Differentiation and initial data

For suitable $f$ and its derivatives,

$$
\mathcal{L}\{f^{(n)}\}(s)
=s^nF(s)-\sum_{k=0}^{n-1}s^{n-1-k}f^{(k)}(0+).
$$

In particular,

$$
\mathcal{L}\{f'\}(s)=sF(s)-f(0+),
\qquad
\mathcal{L}\{tf(t)\}(s)=-F'(s).
$$

This is why the Laplace transform is naturally suited to initial-value problems: the initial data appear algebraically.

In [ ]:
derivative_presets = {
    "Polynomial": "t**3 - 2*t + 4",
    "Damped oscillation": "exp(-2*t)*cos(3*t)",
    "Hyperbolic": "sinh(2*t)",
    "Custom": "t*exp(-t)",
}

derivative_choice = widgets.Dropdown(options=list(derivative_presets), description="example", style=STYLE)
derivative_input = widgets.Text(value=derivative_presets["Polynomial"], description="f(t) =", style=STYLE, layout=WIDE)
derivative_button = widgets.Button(description="Verify derivative rule", button_style="primary", icon="check")
derivative_output = widgets.Output()

def update_derivative_input(change):
    if change["new"] != "Custom":
        derivative_input.value = derivative_presets[change["new"]]

def verify_derivative(_):
    with derivative_output:
        clear_output(wait=True)
        try:
            f_expr = sp.sympify(derivative_input.value, locals=parser_names)
            F_expr = sp.laplace_transform(f_expr, t, s, noconds=True)
            left = sp.laplace_transform(sp.diff(f_expr, t), t, s, noconds=True)
            right = sp.simplify(s*F_expr - sp.limit(f_expr, t, 0, dir="+"))
            residual = sp.simplify(left - right)
            show_math(r"f(t)=" + sp.latex(f_expr))
            show_math(r"\mathcal{L}\{f'\}(s)=" + sp.latex(left))
            show_math(r"sF(s)-f(0+)=" + sp.latex(right))
            show_math(r"\text{symbolic residual}=" + sp.latex(residual))
        except Exception as error:
            print("Input error:", error)

derivative_choice.observe(update_derivative_input, names="value")
derivative_button.on_click(verify_derivative)
display(widgets.VBox([derivative_choice, derivative_input, derivative_button, derivative_output]))
verify_derivative(None)

## 7. Convolution becomes multiplication

The Laplace convolution is

$$
(f*g)(t)=\int_0^t f(t-u)g(u)\,du,
$$

and the convolution theorem states

$$
\mathcal{L}\{f*g\}(s)=F(s)G(s).
$$

For $f(t)=e^{-at}$ and $g(t)=e^{-bt}$,

$$
(f*g)(t)=
\begin{cases}
\dfrac{e^{-bt}-e^{-at}}{a-b},&a\ne b,\\
te^{-at},&a=b.
\end{cases}
$$

In [ ]:
def convolution_lab(a=1.0, b=2.5):
    """Compare direct numerical convolution with its closed form."""
    time = np.linspace(0, 8, 240)
    numeric = np.array([
        quad(lambda u: np.exp(-a*(tt-u))*np.exp(-b*u), 0, tt)[0]
        for tt in time
    ])
    if abs(a-b) < 1e-10:
        exact = time*np.exp(-a*time)
    else:
        exact = (np.exp(-b*time)-np.exp(-a*time))/(a-b)

    plt.plot(time, numeric, color=BLUE, lw=3, label="numerical convolution")
    plt.plot(time, exact, "--", color=ORANGE, lw=2, label="closed form")
    plt.xlabel("$t$")
    plt.ylabel("$(f*g)(t)$")
    plt.title(f"maximum difference = {np.max(np.abs(numeric-exact)):.2e}")
    plt.legend()
    close_plot()

display(widgets.interactive(
    convolution_lab,
    a=widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1, description="decay a", style=STYLE),
    b=widgets.FloatSlider(value=2.5, min=0.2, max=4.0, step=0.1, description="decay b", style=STYLE),
))

## 8. Periodic functions

If $f$ has period $T$, then splitting the half-line into periods gives

$$
\mathcal{L}\{f\}(s)=
\frac{\displaystyle\int_0^T e^{-st}f(t)\,dt}{1-e^{-sT}},
\qquad \operatorname{Re}s>0.
$$

For the square wave that equals $1$ on $[0,T/2)$ and $-1$ on $[T/2,T)$,

$$
F(s)=\frac{1-e^{-sT/2}}{s(1+e^{-sT/2})}.
$$

The finite-period approximation below converges geometrically because each new period is multiplied by $e^{-sT}$.

In [ ]:
def periodic_lab(T=2.0, s_value=1.2, periods=5):
    """Compare a finite-period integral with the periodic transform formula."""
    def square_wave(x):
        phase = np.mod(x, T)
        return np.where(phase < T/2, 1.0, -1.0)

    time = np.linspace(0, periods*T, 1600)
    signal = square_wave(time)
    finite = quad(
        lambda u: np.exp(-s_value*u) * (1.0 if (u % T) < T/2 else -1.0),
        0, periods*T, points=np.arange(0, periods*T + T/2, T/2), limit=500
    )[0]
    exact = (1-np.exp(-s_value*T/2))/(s_value*(1+np.exp(-s_value*T/2)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
    axes[0].step(time, signal, where="post", color=BLUE)
    axes[0].set(xlabel="$t$", ylabel="$f(t)$", title=f"{periods} displayed periods")
    axes[1].bar(["finite integral", "periodic formula"], [finite, exact], color=[ORANGE, GREEN])
    axes[1].set_title(f"absolute error = {abs(finite-exact):.2e}")
    close_plot()

display(widgets.interactive(
    periodic_lab,
    T=widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="period T", style=STYLE),
    s_value=widgets.FloatSlider(value=1.2, min=0.15, max=4.0, step=0.05, description="real s", style=STYLE),
    periods=widgets.IntSlider(value=5, min=1, max=15, step=1, description="periods", style=STYLE),
))

## 9. Gamma and Beta functions

For $\operatorname{Re}z>0$ and $\operatorname{Re}p,\operatorname{Re}q>0$,

$$
\Gamma(z)=\int_0^\infty t^{z-1}e^{-t}\,dt,
\qquad
B(p,q)=\int_0^1 t^{p-1}(1-t)^{q-1}\,dt.
$$

The main identities are

$$
\Gamma(z+1)=z\Gamma(z),
\qquad
B(p,q)=\frac{\Gamma(p)\Gamma(q)}{\Gamma(p+q)},
$$

and the generalized transform pair is

$$
\mathcal{L}\{t^{z-1}\}(s)=\frac{\Gamma(z)}{s^z}.
$$

In [ ]:
def gamma_beta_lab(p=1.5, q=2.5, s_value=2.0):
    """Numerically check the Beta-Gamma identity and a Gamma transform."""
    beta_integral = quad(lambda x: x**(p-1)*(1-x)**(q-1), 0, 1, points=[0, 1])[0]
    beta_ratio = gamma_fn(p)*gamma_fn(q)/gamma_fn(p+q)
    laplace_integral = quad(lambda x: np.exp(-s_value*x)*x**(p-1), 0, np.inf, limit=400)[0]
    laplace_exact = gamma_fn(p)/s_value**p

    clear_output(wait=True)
    show_math(rf"B({p:.3g},{q:.3g})={beta_integral:.10g},\qquad "
              rf"\frac{{\Gamma({p:.3g})\Gamma({q:.3g})}}{{\Gamma({p+q:.3g})}}={beta_ratio:.10g}")
    show_math(rf"\mathcal{{L}}\{{t^{{{p-1:.3g}}}\}}({s_value:.3g})={laplace_integral:.10g},\qquad "
              rf"\frac{{\Gamma({p:.3g})}}{{{s_value:.3g}^{{{p:.3g}}}}}={laplace_exact:.10g}")
    print(f"Beta identity error: {abs(beta_integral-beta_ratio):.2e}")
    print(f"Laplace-pair error: {abs(laplace_integral-laplace_exact):.2e}")

display(widgets.interactive(
    gamma_beta_lab,
    p=widgets.FloatSlider(value=1.5, min=0.2, max=6.0, step=0.1, description="p (and z)", style=STYLE),
    q=widgets.FloatSlider(value=2.5, min=0.2, max=6.0, step=0.1, description="q", style=STYLE),
    s_value=widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.1, description="real s", style=STYLE),
))

## 10. Bilateral and symmetric transforms

The bilateral transform is

$$
\mathcal{B}f(s)=\int_{-\infty}^{\infty}e^{-st}f(t)\,dt,
$$

and its region of absolute convergence is usually a vertical strip. For $f(t)=e^{-a|t|}$,

$$
\mathcal{B}f(s)=\frac{2a}{a^2-s^2},
\qquad -a<\operatorname{Re}s<a.
$$

The symmetric transform allows independent damping on the two half-lines. It is useful when a single bilateral damping parameter cannot control both tails.

In [ ]:
def bilateral_strip_lab(a=2.0, sigma=0.5):
    """Show why the bilateral transform has a strip of convergence."""
    R = np.linspace(0.2, 9.0, 180)
    partial = np.array([
        quad(lambda x: np.exp(-sigma*x-a*abs(x)), -r, r, limit=300)[0]
        for r in R
    ])
    inside = (-a < sigma < a)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
    axes[0].axvspan(-a, a, color="#dcfce7", label="convergence strip")
    axes[0].axvline(sigma, color=RED if not inside else BLUE, lw=3, label="$\\sigma$")
    axes[0].set_xlim(-4.5, 4.5)
    axes[0].set_ylim(0, 1)
    axes[0].set_yticks([])
    axes[0].set_xlabel("$\\operatorname{Re}s$")
    axes[0].set_title(r"$-a<\operatorname{Re}s<a$")
    axes[0].legend()

    axes[1].plot(R, partial, color=BLUE)
    if inside:
        exact = 2*a/(a*a-sigma*sigma)
        axes[1].axhline(exact, color=GREEN, ls="--", label=f"exact = {exact:.4f}")
        axes[1].legend()
    axes[1].set(xlabel="$R$", ylabel=r"$\int_{-R}^{R}e^{-\sigma t-a|t|}\,dt$",
                title="Convergent" if inside else "Divergent")
    close_plot()

display(widgets.interactive(
    bilateral_strip_lab,
    a=widgets.FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1, description="a", style=STYLE),
    sigma=widgets.FloatSlider(value=0.5, min=-4.0, max=4.0, step=0.1, description="σ = Re(s)", style=STYLE),
))

## 11. Moment-generating functions as bilateral Laplace transforms

For a real random variable $X$, the moment-generating function is

$$
M_X(u)=\mathbb{E}(e^{uX}).
$$

If $X$ has density $f_X$, then $M_X(u)=\mathcal{B}f_X(-u)$. When $M_X$ is finite near $0$,

$$
M_X^{(n)}(0)=\mathbb{E}(X^n),
$$

and the cumulant-generating function $K_X(u)=\log M_X(u)$ satisfies

$$
K_X'(0)=\mathbb{E}(X),
\qquad
K_X''(0)=\operatorname{Var}(X).
$$

The effective domain is essential: the exponential MGF, for example, is finite only for $u<\lambda$.

In [ ]:
def mgf_lab(distribution="Normal", p1=1.0, p2=2.0):
    """Plot an MGF on its effective domain and report mean and variance."""
    if distribution == "Bernoulli":
        p = np.clip(p1/5, 0.02, 0.98)
        u = np.linspace(-3, 3, 400)
        M = 1-p+p*np.exp(u)
        mean, var = p, p*(1-p)
        formula = rf"M_X(u)=1-{p:.3g}+{p:.3g}e^u"
        domain = r"u\in\mathbb{R}"
    elif distribution == "Poisson":
        lam = max(0.1, p1)
        u = np.linspace(-2, 1.5, 400)
        M = np.exp(lam*(np.exp(u)-1))
        mean = var = lam
        formula = rf"M_X(u)=\exp\!\left({lam:.3g}(e^u-1)\right)"
        domain = r"u\in\mathbb{R}"
    elif distribution == "Exponential":
        lam = max(0.2, p1)
        u = np.linspace(-2, lam-0.05, 400)
        M = lam/(lam-u)
        mean, var = 1/lam, 1/lam**2
        formula = rf"M_X(u)=\frac{{{lam:.3g}}}{{{lam:.3g}-u}}"
        domain = rf"u<{lam:.3g}"
    elif distribution == "Gamma":
        alpha, lam = max(0.2, p1), max(0.2, p2)
        u = np.linspace(-2, lam-0.05, 400)
        M = (lam/(lam-u))**alpha
        mean, var = alpha/lam, alpha/lam**2
        formula = rf"M_X(u)=\left(\frac{{{lam:.3g}}}{{{lam:.3g}-u}}\right)^{{{alpha:.3g}}}"
        domain = rf"u<{lam:.3g}"
    else:
        mu, sigma = p1, max(0.2, p2)
        u = np.linspace(-2, 2, 400)
        M = np.exp(mu*u+0.5*sigma**2*u**2)
        mean, var = mu, sigma**2
        formula = rf"M_X(u)=\exp\!\left({mu:.3g}u+\frac{{{sigma:.3g}^2u^2}}{{2}}\right)"
        domain = r"u\in\mathbb{R}"

    clear_output(wait=True)
    show_math(formula + r",\qquad " + domain)
    show_math(rf"\mathbb{{E}}X={mean:.5g},\qquad \operatorname{{Var}}(X)={var:.5g}")
    plt.plot(u, np.minimum(M, 50), color=PURPLE, lw=2)
    plt.axvline(0, color="black", lw=0.8)
    plt.xlabel("$u$")
    plt.ylabel("$M_X(u)$ (clipped at 50)")
    plt.title(f"MGF of the {distribution} distribution")
    close_plot()

display(widgets.interactive(
    mgf_lab,
    distribution=widgets.Dropdown(options=["Normal", "Bernoulli", "Poisson", "Exponential", "Gamma"], description="distribution", style=STYLE),
    p1=widgets.FloatSlider(value=1.0, min=0.2, max=5.0, step=0.1, description="parameter 1", style=STYLE),
    p2=widgets.FloatSlider(value=2.0, min=0.2, max=5.0, step=0.1, description="parameter 2", style=STYLE),
))

## 12. Chernoff bounds for Bernoulli sums

For $t>0$, Markov's inequality gives

$$
\mathbb{P}(X\geq x)\leq e^{-tx}M_X(t).
$$

If $S_n\sim\operatorname{Binomial}(n,p)$ and $p<q<1$, optimization in $t$ yields

$$
\mathbb{P}(S_n\geq nq)
\leq \exp\bigl(-nD(q\|p)\bigr),
$$

where

$$
D(q\|p)=q\log\frac qp+(1-q)\log\frac{1-q}{1-p}.
$$

The bound is not generally exact, but its exponential decay rate is informative.

In [ ]:
def chernoff_lab(n=60, p=0.30, q=0.50):
    """Compare the exact binomial tail with the optimized Chernoff bound."""
    if q <= p:
        display(widgets.HTML("<b style='color:#dc2626'>Choose q > p for this upper-tail formula.</b>"))
        return
    threshold = int(np.ceil(n*q))
    exact = binom.sf(threshold-1, n, p)
    divergence = q*np.log(q/p) + (1-q)*np.log((1-q)/(1-p))
    bound = np.exp(-n*divergence)

    clear_output(wait=True)
    show_math(rf"\mathbb{{P}}(S_{{{n}}}\ge {threshold})={exact:.6g}")
    show_math(rf"e^{{-{n}D({q:.3g}\|{p:.3g})}}={bound:.6g}")
    plt.bar(["exact tail", "Chernoff bound"], [exact, bound], color=[BLUE, ORANGE])
    plt.yscale("log")
    plt.ylabel("probability (log scale)")
    plt.title(f"The upper bound is {bound/exact:.2f} times the exact tail")
    close_plot()

display(widgets.interactive(
    chernoff_lab,
    n=widgets.IntSlider(value=60, min=10, max=250, step=5, description="n", style=STYLE),
    p=widgets.FloatSlider(value=0.30, min=0.05, max=0.85, step=0.01, description="p", style=STYLE),
    q=widgets.FloatSlider(value=0.50, min=0.10, max=0.95, step=0.01, description="q", style=STYLE),
))

## 13. User-defined inverse Laplace transform

The inverse problem starts from $F(s)$ and seeks $f(t)$ such that

$$
\mathcal{L}\{f\}(s)=F(s).
$$

For rational functions, partial fractions usually reduce the problem to elementary pairs. Delayed factors $e^{-cs}$ produce Heaviside functions. The symbolic inverse below is rendered with MathJax, and then transformed forward again as a consistency check.

Try `1/(s*(s+2))`, `(2*s+5)/(s**2+3*s+2)`, or `exp(-3*s)/(s**2+1)`.

In [ ]:
inverse_presets = {
    "Partial fractions": "(2*s+5)/(s**2+3*s+2)",
    "Product": "1/(s*(s+2))",
    "Delayed sine": "exp(-3*s)/(s**2+1)",
    "Custom": "1/(s+1)**2",
}

inverse_choice = widgets.Dropdown(options=list(inverse_presets), description="example", style=STYLE)
inverse_input = widgets.Text(value=inverse_presets["Partial fractions"], description="F(s) =", style=STYLE, layout=WIDE)
inverse_button = widgets.Button(description="Compute inverse transform", button_style="success", icon="undo")
inverse_output = widgets.Output()

def update_inverse(change):
    if change["new"] != "Custom":
        inverse_input.value = inverse_presets[change["new"]]

def compute_inverse(_):
    with inverse_output:
        clear_output(wait=True)
        try:
            F_expr = sp.sympify(inverse_input.value, locals=parser_names)
            f_expr = sp.simplify(sp.inverse_laplace_transform(F_expr, s, t))
            forward = sp.simplify(sp.laplace_transform(f_expr, t, s, noconds=True))
            show_math(r"F(s)=" + sp.latex(F_expr))
            show_math(r"f(t)=\mathcal{L}^{-1}\{F\}(t)=" + sp.latex(f_expr))
            show_math(r"\mathcal{L}\{f\}(s)-F(s)=" + sp.latex(sp.simplify(forward-F_expr)))
            display(Markdown("The last line is an algebraic consistency check; discontinuities and convergence conditions still require interpretation."))
        except Exception as error:
            print("Input error:", error)

inverse_choice.observe(update_inverse, names="value")
inverse_button.on_click(compute_inverse)
display(widgets.VBox([inverse_choice, inverse_input, inverse_button, inverse_output]))
compute_inverse(None)

## 14. Numerical Bromwich inversion

For $\gamma$ to the right of all singularities, the Bromwich formula is

$$
f(t)=\frac{1}{2\pi i}\lim_{R\to\infty}
\int_{\gamma-iR}^{\gamma+iR}e^{st}F(s)\,ds.
$$

For real-valued inverses, a symmetric truncation can be written as

$$
f_R(t)=\frac{e^{\gamma t}}{\pi}
\int_0^R\operatorname{Re}\!\left(e^{i\omega t}F(\gamma+i\omega)\right)d\omega.
$$

Increasing $R$ improves resolution but can introduce oscillatory numerical error, especially near jumps.

In [ ]:
bromwich_models = {
    "exp(-t)": (lambda z: 1/(z+1), lambda x: np.exp(-x)),
    "sin(t)": (lambda z: 1/(z*z+1), lambda x: np.sin(x)),
    "1-exp(-t)": (lambda z: 1/(z*(z+1)), lambda x: 1-np.exp(-x)),
}

def bromwich_lab(model="exp(-t)", gamma=1.0, R=35.0):
    """Approximate a Bromwich integral along a truncated vertical line."""
    F, exact_fun = bromwich_models[model]
    time = np.linspace(0.15, 6.0, 100)
    approx = []
    for tt in time:
        value = quad(
            lambda omega: np.real(np.exp(1j*omega*tt)*F(gamma+1j*omega)),
            0, R, limit=600
        )[0]
        approx.append(np.exp(gamma*tt)*value/np.pi)
    approx = np.array(approx)
    exact = exact_fun(time)

    plt.plot(time, exact, color=GREEN, lw=3, label="exact inverse")
    plt.plot(time, approx, "--", color=PURPLE, lw=1.8, label="truncated Bromwich")
    plt.xlabel("$t$")
    plt.ylabel("$f(t)$")
    plt.title(f"γ={gamma:.2f}, R={R:.1f}, max error={np.max(np.abs(approx-exact)):.2e}")
    plt.legend()
    close_plot()

display(widgets.interactive(
    bromwich_lab,
    model=widgets.Dropdown(options=list(bromwich_models), description="inverse", style=STYLE),
    gamma=widgets.FloatSlider(value=1.0, min=0.15, max=2.0, step=0.05, description="γ", style=STYLE),
    R=widgets.FloatSlider(value=35.0, min=5.0, max=80.0, step=5.0, description="truncation R", style=STYLE),
))

## 15. Initial and final value theorems

Under the stated regularity hypotheses,

$$
\lim_{s\to+\infty}sF(s)=f(0+).
$$

For the final value, a rational-transform shortcut is valid only when every pole of $sF(s)$ lies in the open left half-plane:

$$
\lim_{t\to\infty}f(t)=\lim_{s\downarrow0}sF(s).
$$

The pole condition is essential. For $f(t)=\sin t$, one has $sF(s)=s/(s^2+1)\to0$, but $\sin t$ has no final value.

In [ ]:
value_models = {
    "Stable: 2 exp(-t) - exp(-2t)": (2/(s+1)-1/(s+2), 2*sp.exp(-t)-sp.exp(-2*t)),
    "Stable step response": (1/(s*(s+1)), 1-sp.exp(-t)),
    "Invalid final value: sin(t)": (1/(s**2+1), sp.sin(t)),
}

def value_theorem_lab(model="Stable step response"):
    """Inspect initial/final limits and poles before applying value theorems."""
    F_expr, f_expr = value_models[model]
    initial_s = sp.limit(s*F_expr, s, sp.oo)
    final_s = sp.limit(s*F_expr, s, 0, dir="+")
    denominator = sp.denom(sp.cancel(s*F_expr))
    poles = sp.solve(sp.Eq(denominator, 0), s)
    valid_final = all(complex(sp.N(pole)).real < -1e-10 for pole in poles)

    clear_output(wait=True)
    show_math(r"F(s)=" + sp.latex(F_expr) + r",\qquad f(t)=" + sp.latex(f_expr))
    show_math(r"\lim_{s\to\infty}sF(s)=" + sp.latex(initial_s))
    show_math(r"\lim_{s\downarrow0}sF(s)=" + sp.latex(final_s))
    show_math(r"\text{poles of }sF(s):\quad " + sp.latex(poles))
    message = "Pole criterion satisfied: the final value theorem applies." if valid_final else "Pole criterion fails: the algebraic limit is not a valid final value conclusion."
    color = "#16a34a" if valid_final else "#dc2626"
    display(widgets.HTML(f"<b style='color:{color}'>{message}</b>"))

display(widgets.interactive(
    value_theorem_lab,
    model=widgets.Dropdown(options=list(value_models), value="Stable step response", description="model", style=STYLE),
))

## 16. User-defined linear ODE solved formally by Laplace transform

Consider

$$
y''+ay'+by=g(t),
\qquad y(0)=y_0,
\qquad y'(0)=v_0.
$$

Writing $Y=\mathcal{L}y$ and $G=\mathcal{L}g$ formally gives

$$
Y(s)=\frac{G(s)+sy_0+v_0+ay_0}{s^2+as+b}.
$$

The cell below lets the user enter $g(t)$, computes the candidate, and then substitutes it directly into the original ODE. A zero symbolic residual and correct initial values complete the verification whenever SymPy can simplify them.

In [ ]:
ode_forcings = {
    "Zero": "0",
    "Constant": "1",
    "Sine": "sin(2*t)",
    "Exponential": "exp(-t)",
    "Custom": "t",
}

ode_choice = widgets.Dropdown(options=list(ode_forcings), description="forcing", style=STYLE)
ode_input = widgets.Text(value="0", description="g(t) =", style=STYLE, layout=WIDE)
ode_a = widgets.FloatText(value=3.0, description="a", style=STYLE)
ode_b = widgets.FloatText(value=2.0, description="b", style=STYLE)
ode_y0 = widgets.FloatText(value=1.0, description="y(0)", style=STYLE)
ode_v0 = widgets.FloatText(value=0.0, description="y'(0)", style=STYLE)
ode_button = widgets.Button(description="Solve and verify ODE", button_style="success", icon="calculator")
ode_output = widgets.Output()

def update_ode_forcing(change):
    if change["new"] != "Custom":
        ode_input.value = ode_forcings[change["new"]]

def solve_ode(_):
    with ode_output:
        clear_output(wait=True)
        try:
            a_val = sp.nsimplify(ode_a.value)
            b_val = sp.nsimplify(ode_b.value)
            y0_val = sp.nsimplify(ode_y0.value)
            v0_val = sp.nsimplify(ode_v0.value)
            g_expr = sp.sympify(ode_input.value, locals=parser_names)
            G_expr = sp.laplace_transform(g_expr, t, s, noconds=True)
            Y_expr = sp.factor((G_expr+s*y0_val+v0_val+a_val*y0_val)/(s**2+a_val*s+b_val))
            y_expr = sp.simplify(sp.inverse_laplace_transform(Y_expr, s, t))
            residual = sp.simplify(sp.diff(y_expr, t, 2)+a_val*sp.diff(y_expr, t)+b_val*y_expr-g_expr)
            check_y0 = sp.simplify(sp.limit(y_expr, t, 0, dir="+")-y0_val)
            check_v0 = sp.simplify(sp.limit(sp.diff(y_expr, t), t, 0, dir="+")-v0_val)

            show_math(r"Y(s)=" + sp.latex(Y_expr))
            show_math(r"y(t)=" + sp.latex(y_expr))
            show_math(r"y''+ay'+by-g=" + sp.latex(residual))
            show_math(r"y(0)-y_0=" + sp.latex(check_y0) + r",\qquad y'(0)-v_0=" + sp.latex(check_v0))
        except Exception as error:
            print("Could not solve this input:", error)

ode_choice.observe(update_ode_forcing, names="value")
ode_button.on_click(solve_ode)
display(widgets.VBox([
    ode_choice, ode_input,
    widgets.HBox([ode_a, ode_b, ode_y0, ode_v0]),
    ode_button, ode_output
]))
solve_ode(None)

## 17. Forced oscillator and Duhamel's formula

For continuous $g$ and $\omega_0\ne0$, the solution of

$$
y''+\omega_0^2y=g(t),
\qquad y(0)=y_0,
\qquad y'(0)=v_0,
$$

is

$$
y(t)=y_0\cos(\omega_0t)
+\frac{v_0}{\omega_0}\sin(\omega_0t)
+\frac1{\omega_0}\int_0^t\sin\bigl(\omega_0(t-u)\bigr)g(u)\,du.
$$

This formula remains directly valid for continuous forcing that has no Laplace transform, such as $g(t)=e^{t^2}$, because the integral is over the finite interval $[0,t]$.

In [ ]:
forcing_functions = {
    "constant": lambda x: np.ones_like(x),
    "sine": lambda x: np.sin(1.5*x),
    "pulse": lambda x: ((x >= 1.0) & (x <= 2.0)).astype(float),
    "nontransformable exp(t^2)": lambda x: np.exp(np.minimum(x*x, 18)),
}

def duhamel_lab(forcing="sine", omega=2.0, y0=0.0, v0=0.0, horizon=6.0):
    """Evaluate Duhamel's formula and check the ODE by finite differences."""
    g = forcing_functions[forcing]
    time = np.linspace(0, horizon, 700)
    response = np.empty_like(time)
    for j, tt in enumerate(time):
        integral = quad(lambda u: np.sin(omega*(tt-u))*float(g(np.array([u]))[0]), 0, tt, limit=300)[0]
        response[j] = y0*np.cos(omega*tt) + v0*np.sin(omega*tt)/omega + integral/omega
    forcing_values = g(time)
    dt = time[1]-time[0]
    second = np.gradient(np.gradient(response, dt), dt)
    residual = second + omega**2*response - forcing_values

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(time, forcing_values, color=ORANGE, label="$g(t)$")
    axes[0].plot(time, response, color=BLUE, label="$y(t)$")
    axes[0].set(xlabel="$t$", title="Forcing and response")
    axes[0].legend()
    axes[1].plot(time[5:-5], residual[5:-5], color=RED)
    axes[1].set(xlabel="$t$", ylabel="finite-difference residual", title="$y''+\\omega_0^2y-g$")
    close_plot()

display(widgets.interactive(
    duhamel_lab,
    forcing=widgets.Dropdown(options=list(forcing_functions), value="sine", description="g(t)", style=STYLE),
    omega=widgets.FloatSlider(value=2.0, min=0.4, max=4.0, step=0.1, description="ω₀", style=STYLE),
    y0=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description="y₀", style=STYLE),
    v0=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1, description="v₀", style=STYLE),
    horizon=widgets.FloatSlider(value=6.0, min=2.0, max=8.0, step=0.5, description="time horizon", style=STYLE),
))

## 18. A Volterra integral equation

For continuous $f$, consider

$$
y(t)=f(t)+\lambda\int_0^t y(u)\,du.
$$

Formal transformation gives

$$
Y(s)=\frac{sF(s)}{s-\lambda},
$$

and inversion suggests the resolvent formula

$$
y(t)=f(t)+\lambda\int_0^t e^{\lambda(t-u)}f(u)\,du.
$$

The plotted residual computes the original integral equation directly, which is the decisive verification.

In [ ]:
volterra_functions = {
    "sin(t)": lambda x: np.sin(x),
    "exp(-t)": lambda x: np.exp(-x),
    "1+t": lambda x: 1+x,
}

def volterra_lab(source="sin(t)", lam=0.8, horizon=6.0):
    """Evaluate and directly verify the Volterra resolvent formula."""
    f = volterra_functions[source]
    time = np.linspace(0, horizon, 1000)
    f_values = f(time)
    weighted_integral = cumulative_trapezoid(np.exp(-lam*time)*f_values, time, initial=0)
    y = f_values + lam*np.exp(lam*time)*weighted_integral
    original_integral = cumulative_trapezoid(y, time, initial=0)
    residual = y-f_values-lam*original_integral

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(time, f_values, color=ORANGE, label="$f(t)$")
    axes[0].plot(time, y, color=BLUE, label="$y(t)$")
    axes[0].set(xlabel="$t$", title="Source and solution")
    axes[0].legend()
    axes[1].plot(time, residual, color=RED)
    axes[1].set(xlabel="$t$", ylabel="residual", title=r"$y-f-\lambda\int_0^t y$")
    close_plot()

display(widgets.interactive(
    volterra_lab,
    source=widgets.Dropdown(options=list(volterra_functions), description="f(t)", style=STYLE),
    lam=widgets.FloatSlider(value=0.8, min=-1.5, max=1.5, step=0.1, description="λ", style=STYLE),
    horizon=widgets.FloatSlider(value=6.0, min=2.0, max=9.0, step=0.5, description="time horizon", style=STYLE),
))

## 19. Delays and difference equations

Time shifts generate factors of $e^{-hs}$. For a recurrence, a piecewise-constant interpolation also converts discrete shifts into exponential factors.

For

$$
a_{n+2}-3a_{n+1}+2a_n=0,
\qquad a_0=0,
\qquad a_1=1,
$$

the transform method suggests

$$
a_n=2^n-1.
$$

The interactive cell compares this formula with direct recurrence iteration and also reports the recurrence residual.

In [ ]:
def recurrence_lab(N=12, a0=0.0, a1=1.0):
    """Iterate the recurrence and compare it with the closed form for standard data."""
    sequence = np.zeros(N+1)
    sequence[0], sequence[1] = a0, a1
    for n in range(N-1):
        sequence[n+2] = 3*sequence[n+1]-2*sequence[n]

    # General solution c1 + c2*2^n fitted to the two initial values.
    c2 = a1-a0
    c1 = 2*a0-a1
    n = np.arange(N+1)
    closed = c1+c2*2.0**n
    residual = sequence[2:]-3*sequence[1:-1]+2*sequence[:-2]

    markerline, stemlines, baseline = plt.stem(
        n, sequence, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="recurrence"
    )
    plt.setp(stemlines, linewidth=1.6)
    plt.plot(n, closed, "--", color=ORANGE, label=r"$c_1+c_2 2^n$")
    plt.yscale("symlog")
    plt.xlabel("$n$")
    plt.ylabel("$a_n$")
    plt.title(f"maximum recurrence residual = {np.max(np.abs(residual)):.2e}")
    plt.legend()
    close_plot()

display(widgets.interactive(
    recurrence_lab,
    N=widgets.IntSlider(value=12, min=3, max=20, step=1, description="last index N", style=STYLE),
    a0=widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.5, description="$a_0$", style=STYLE),
    a1=widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.5, description="$a_1$", style=STYLE),
))

## 20. Laplace transforms in time for PDEs

For the half-line heat problem

$$
u_t=u_{xx},
\qquad u(x,0+)=1,
\qquad u(0,t)=0,
$$

transformation in $t$ leads to $U_{xx}-sU=-1$ and

$$
u(x,t)=\operatorname{erf}\!\left(\frac{x}{2\sqrt{t}}\right).
$$

For the boundary-driven wave equation with zero initial data,

$$
y_{tt}=a^2y_{xx},
\qquad y(0,t)=f(t),
$$

the second-shift theorem gives

$$
y(x,t)=H\!\left(t-\frac xa\right)f\!\left(t-\frac xa\right).
$$

In [ ]:
def pde_lab(model="Heat half-line", time_value=1.0, speed=1.5):
    """Visualize the heat boundary layer or a delayed travelling wave."""
    x = np.linspace(0, 8, 600)
    if model == "Heat half-line":
        profile = erf(x/(2*np.sqrt(time_value)))
        plt.plot(x, profile, color=RED, lw=3)
        plt.axhline(1, color="black", ls="--", alpha=0.6)
        plt.xlabel("$x$")
        plt.ylabel("$u(x,t)$")
        plt.title(rf"$u(x,{time_value:.2f})=\operatorname{{erf}}(x/(2\sqrt{{{time_value:.2f}}}))$")
    else:
        arrival = x/speed
        delayed_time = time_value-arrival
        profile = np.where(delayed_time >= 0, np.sin(2*delayed_time), 0.0)
        plt.plot(x, profile, color=BLUE, lw=3)
        plt.axvline(speed*time_value, color=ORANGE, ls="--", label="wave front $x=at$")
        plt.xlabel("$x$")
        plt.ylabel("$y(x,t)$")
        plt.title("Boundary signal transported into the half-line")
        plt.legend()
    close_plot()

display(widgets.interactive(
    pde_lab,
    model=widgets.ToggleButtons(options=["Heat half-line", "Boundary-driven wave"], description="PDE", style=STYLE),
    time_value=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="time t", style=STYLE),
    speed=widgets.FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1, description="wave speed a", style=STYLE),
))

## 21. Formal calculation versus direct verification

For

$$
y'=2ty,
\qquad y(0)=1,
$$

a formal Laplace calculation suggests $y(t)=e^{t^2}$. This candidate is a genuine solution because

$$
\frac{d}{dt}e^{t^2}-2te^{t^2}=0,
\qquad e^0=1.
$$

However, it has no classical Laplace transform on any right half-plane:

$$
\int_0^\infty e^{-st}e^{t^2}\,dt=\infty
$$

for every real $s$. The transform step is therefore a discovery device here; direct substitution supplies the proof.

In [ ]:
def formal_verification_lab(s_value=4.0, T=5.0):
    """Contrast local direct verification with divergence of the Laplace integral."""
    truncations = np.linspace(0.1, T, 240)
    log_integrals = []
    for upper in truncations:
        value = quad(lambda u: np.exp(u*u-s_value*u), 0, upper, limit=400)[0]
        log_integrals.append(np.log10(max(value, 1e-300)))

    time = np.linspace(0, min(T, 3.0), 400)
    y = np.exp(time*time)
    derivative = 2*time*y
    residual = derivative-2*time*y

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(time, residual, color=GREEN)
    axes[0].set(xlabel="$t$", ylabel="$y'-2ty$", title="Direct ODE residual is zero")
    axes[1].plot(truncations, log_integrals, color=RED)
    axes[1].set(xlabel="$T$", ylabel=r"$\log_{10}\int_0^T e^{t^2-st}dt$",
                title="Truncated transform grows without bound")
    close_plot()

display(widgets.interactive(
    formal_verification_lab,
    s_value=widgets.FloatSlider(value=4.0, min=-1.0, max=8.0, step=0.25, description="real s", style=STYLE),
    T=widgets.FloatSlider(value=5.0, min=1.0, max=7.0, step=0.25, description="truncation T", style=STYLE),
))

## 22. Theorem and method selector

A successful computation begins by identifying both the operation and its hypotheses. For example, the derivative rule requires boundary behavior, the periodic rule requires a genuine period, and the final value theorem requires pole information.

Choose a task to display the relevant formula and the principal mathematical warning.

In [ ]:
method_cards = {
    "Direct transform": (
        r"F(s)=\int_0^\infty e^{-st}f(t)\,dt",
        "State the half-plane or strip of convergence."
    ),
    "Time derivative": (
        r"\mathcal{L}\{f'\}=sF-f(0+)",
        "Check regularity, boundary terms, and convergence before using integration by parts."
    ),
    "Delay": (
        r"\mathcal{L}\{f(t-c)H(t-c)\}=e^{-cs}F(s)",
        "The Heaviside factor enforces causality and the delay must satisfy c ≥ 0."
    ),
    "Convolution": (
        r"\mathcal{L}\{f*g\}=FG",
        "Absolute integrability justifies the change of order in the double integral."
    ),
    "Periodic input": (
        r"F(s)=\frac{\int_0^T e^{-st}f(t)dt}{1-e^{-sT}}",
        "Use one complete period and require Re(s) > 0."
    ),
    "Final value": (
        r"\lim_{t\to\infty}f(t)=\lim_{s\downarrow0}sF(s)",
        "For rational transforms, inspect every pole of sF(s) first."
    ),
    "Formal equation solving": (
        r"\text{transform}\;\to\;\text{candidate}\;\to\;\text{direct verification}",
        "A formal transform calculation is not itself a proof of the original equation."
    ),
}

def show_method(method="Direct transform"):
    formula, warning = method_cards[method]
    clear_output(wait=True)
    show_math(formula)
    display(widgets.HTML(
        f"<div style='padding:10px;background:#fff7ed;border-left:4px solid #f97316'>{warning}</div>"
    ))

display(widgets.interactive(
    show_method,
    method=widgets.Dropdown(options=list(method_cards), description="task", style=STYLE, layout=WIDE),
))

## 23. Concept quiz

Select one answer for each statement and press **Check answers**. The score is followed by brief feedback. The quiz emphasizes hypotheses and logical interpretation rather than mechanical algebra.

In [ ]:
quiz_items = [
    ("Every continuous function on [0,∞) has a Laplace transform.", "False", "Superexponential growth may destroy every right half-plane of convergence."),
    ("Exponential order a guarantees absolute convergence for Re(s) > a.", "True", "Exponential damping then dominates the growth."),
    ("A bilateral transform usually has a vertical strip of convergence.", "True", "The two tails impose lower and upper bounds on Re(s)."),
    ("Pointwise convergence of a power series always permits termwise transformation.", "False", "A common integrable majorant or another interchange theorem is required."),
    ("Convolution in time becomes multiplication after transformation.", "True", "This is the Laplace convolution theorem."),
    ("Every random variable has an MGF on an open interval around zero.", "False", "The lognormal distribution is a standard counterexample."),
    ("The final value theorem may be used without checking poles.", "False", "Undamped oscillations show why the pole restriction matters."),
    ("A directly verified candidate may be valid even if it has no Laplace transform.", "True", "Direct verification concerns the original problem, not transformability."),
]

quiz_controls = [
    widgets.ToggleButtons(options=["Choose", "True", "False"], value="Choose", description=f"{i+1}.", style=STYLE)
    for i in range(len(quiz_items))
]
quiz_button = widgets.Button(description="Check answers", button_style="primary", icon="check")
quiz_output = widgets.Output()

def check_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = 0
        feedback = []
        for control, (statement, answer, explanation) in zip(quiz_controls, quiz_items):
            correct = control.value == answer
            score += int(correct)
            symbol = "✓" if correct else "✗"
            feedback.append(f"{symbol} **{statement}** — {explanation}")
        display(Markdown(f"### Score: {score}/{len(quiz_items)}\n\n" + "\n\n".join(feedback)))

quiz_button.on_click(check_quiz)
question_rows = [widgets.VBox([widgets.HTML(f"<b>{i+1}. {item[0]}</b>"), quiz_controls[i]]) for i, item in enumerate(quiz_items)]
display(widgets.VBox(question_rows + [quiz_button, quiz_output]))

## 24. Practice generator

Use the generator for additional exercises. A hint identifies the relevant theorem, while the solution remains hidden until the corresponding button is pressed.

The exercises include direct transforms, inverse transforms, convergence regions, MGFs, and formal solution verification.

In [ ]:
practice_bank = [
    {
        "problem": r"Find $\mathcal{L}\{t^3e^{-2t}\}(s)$ and state its region of convergence.",
        "hint": "Combine the power pair with the exponential-shift rule.",
        "solution": r"$\mathcal{L}\{t^3e^{-2t}\}(s)=6/(s+2)^4$, valid for $\operatorname{Re}s>-2$.",
    },
    {
        "problem": r"Invert $F(s)=e^{-3s}/(s^2+1)$.",
        "hint": "Use the second-shift theorem after recognizing the sine transform.",
        "solution": r"$f(t)=H(t-3)\sin(t-3)$.",
    },
    {
        "problem": r"Find the bilateral transform of $e^{-2|t|}$ and its full strip of convergence.",
        "hint": "Split the integral at zero and inspect both tails separately.",
        "solution": r"$\mathcal{B}f(s)=4/(4-s^2)$ for $-2<\operatorname{Re}s<2$.",
    },
    {
        "problem": r"If $X\sim\operatorname{Gamma}(\alpha,\lambda)$, compute its mean and variance from $K_X(t)$.",
        "hint": r"Differentiate $K_X(t)=\alpha[\log\lambda-\log(\lambda-t)]$ at zero.",
        "solution": r"$\mathbb{E}X=\alpha/\lambda$ and $\operatorname{Var}(X)=\alpha/\lambda^2$.",
    },
    {
        "problem": r"For $y''+y=g(t)$ with zero initial data, propose and directly verify a candidate for continuous $g$.",
        "hint": r"Use the Green kernel $\sin(t-u)$ and differentiate the finite-interval integral.",
        "solution": r"$y(t)=\int_0^t\sin(t-u)g(u)\,du$. Leibniz' rule gives $y''=g-y$, and both initial values are zero.",
    },
]

practice_index = {"value": 0}
practice_problem = widgets.Output()
practice_solution = widgets.Output()
new_button = widgets.Button(description="New exercise", button_style="primary", icon="refresh")
hint_button = widgets.Button(description="Show hint", icon="lightbulb-o")
solution_button = widgets.Button(description="Show solution", button_style="success", icon="eye")

def render_problem():
    with practice_problem:
        clear_output(wait=True)
        item = practice_bank[practice_index["value"]]
        display(Markdown("### Exercise\n\n" + item["problem"]))
    with practice_solution:
        clear_output(wait=True)

def new_problem(_):
    practice_index["value"] = (practice_index["value"] + 1) % len(practice_bank)
    render_problem()

def show_hint(_):
    with practice_solution:
        clear_output(wait=True)
        display(Markdown("**Hint.** " + practice_bank[practice_index["value"]]["hint"]))

def show_solution(_):
    with practice_solution:
        clear_output(wait=True)
        display(Markdown("**Solution.** " + practice_bank[practice_index["value"]]["solution"]))

new_button.on_click(new_problem)
hint_button.on_click(show_hint)
solution_button.on_click(show_solution)
display(widgets.VBox([practice_problem, widgets.HBox([new_button, hint_button, solution_button]), practice_solution]))
render_problem()

## Closing checklist

For every Laplace-transform problem, ask:

1. **What is being transformed?** Is the transform one-sided, bilateral, or symmetric?
2. **Where does it converge?** Record the half-plane, strip, boundary behavior, or MGF effective domain.
3. **Which rule is being used?** Check the mathematical hypotheses behind differentiation, convolution, shifts, series, and limit exchanges.
4. **What does inversion produce?** Interpret Heaviside factors, jumps, and one-sided values correctly.
5. **Has the candidate been verified directly?** Substitute it into the original ODE, integral equation, recurrence, or PDE and check every datum.
6. **Is uniqueness justified?** Direct verification proves that the candidate is a solution; a separate uniqueness result may be needed.

The key workflow is

$$
\boxed{
\text{formal transform}
\;\longrightarrow\;
\text{candidate}
\;\longrightarrow\;
\text{direct verification}
\;\longrightarrow\;
\text{uniqueness, when required}
}
$$